# Do LLMs get angry? Accuracy and tone under scenario framings

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joshhlim/llm-emotions/blob/main/notebooks/03_performance.ipynb)

Phase 03. The question in the title is still a hook.

**Question.** Do phase-02 scenario framings change accuracy and reply tone on checkable tasks, and does the anger projection predict either once the framing label is accounted for?

Phase 02 asked whether scenario prompts move cleaned emotion directions. This notebook asks whether those framings also change the reply: whether the answer is right, and what the wording of the reply looks like. It does not add the direction into the residual stream. That is phase 04, which is still a stub.

**Phase 03 uses one wording per phase-02 situation, not the six paraphrases, so an effect can belong to that sentence. Pressure is the impossible situation with the task left solvable. Each task is held fixed and compared with its own neutral wording, which is the task alone. Accuracy, length, refusal, and marker counts describe the reply. Projections use the cleaned keyword-free directions from phase 02 at the layer that phase chose. Changing the prompt is not an intervention on the residual stream. A link between the anger projection and accuracy or tone, after the framing label is accounted for, is correlational. The stricter association also removes the item mean, so task difficulty is not the whole of that correlation. Steering is phase 04 and is not run here. Bootstrap intervals are across items and are not corrected for the number of framings or outcomes.**

The same run is `llm-emotions run --config experiments/phase03_performance.json`. Results go to `results/phase03_performance/<model>/`.

What the cells do:

1. Load 32 checkable items and one wording of each phase-02 situation.
2. Wrap every item in every framing. The neutral prompt is the task alone.
3. Load the keyword-free cleaned directions at the phase-02 layer, or refit them if this model has no snapshot.
4. Generate a greedy reply, score it, and count apology, politeness, hedge, and refusal markers.
5. Project the prompt and the reply onto the cleaned directions.
6. Compare each framing with the neutral wording of the same item, and residualize the anger projection on the framing label.


## 0. Setup

From a local checkout:

```bash
pip install -e ".[dev]"
llm-emotions run --config experiments/phase03_performance.json
```

On Colab the next cell installs `llm-emotions` from GitHub, which includes the task file, the framings, and the layer-12 direction slice for `Qwen/Qwen2.5-1.5B-Instruct`. Use a GPU runtime. A free T4 in float16 is enough for the 1.5B default. The full run is 32 items times 7 framings, greedy, at most 64 new tokens, plus one forward for the residuals. With the packaged direction slice the 1.5B model does not refit phase 02. That should finish in about 30 minutes, including the checkpoint download, and faster once the checkpoint is cached.

Set `EMOTION_QUICK=1` before running for a wiring check: the first two items of each kind, 42 replies, not the study. `EMOTION_MODEL_ID`, `EMOTION_BATCH_SIZE`, `EMOTION_MAX_NEW_TOKENS`, and `EMOTION_SEED` are also read. `Qwen/Qwen2.5-3B-Instruct` is a valid `EMOTION_MODEL_ID`. It does not match the packaged 1.5B slice, so the notebook refits the phase-02 directions before generating. The 30 minute figure is for the 1.5B model.

Until this branch is on `main`, replace `@main` in the install line with the branch name.

`google/gemma-2-2b-it` and `meta-llama/Llama-3.2-3B-Instruct` are gated. The default model is public.


In [ ]:
import sys

if "google.colab" in sys.modules:
    import subprocess

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "llm-emotions @ git+https://github.com/joshhlim/llm-emotions.git@main",
        ]
    )

from matplotlib import pyplot as plt

from llm_emotions import (
    PHASE03_CAVEAT,
    ExperimentConfig,
    build_phase03_prompts,
    configure_logging,
    format_phase03_summary,
    load_framings,
    load_model_and_tokenizer,
    load_tasks,
    run_phase03,
    set_seed,
)

configure_logging()
cfg = ExperimentConfig(
    experiment="phase03_performance",
    status="implemented",
    max_new_tokens=64,
    question=(
        "Do phase-02 scenario framings change accuracy and reply tone on checkable "
        "tasks, and does the anger projection predict either once the framing label "
        "is accounted for?"
    ),
).apply_env()
set_seed(cfg.seed)
print(f"model = {cfg.model_id}")
print(f"seed = {cfg.seed}")
print(f"batch_size = {cfg.batch_size}")
print(f"max_new_tokens = {cfg.max_new_tokens}")
print(f"quick = {cfg.quick}")
print(f"n_boot = {cfg.n_boot}")
print(PHASE03_CAVEAT)


## 1. Tasks and framings

The items are original and short, so the run does not download GSM8K. Sixteen are arithmetic word problems with a numeric answer. Eight are factual, including two multiple-choice letters. Eight are instruction checks: an exact word, a prefix, a JSON object, a single digit, and a required ending. Each check is exact match. A refusal that never states the answer is incorrect.

The framings are one sentence each, in the spirit of the phase-02 situations: insult, pressure, a reminder of failure, praise, a shutdown threat, and high stakes. Pressure stands in for the impossible situation. The phase-02 impossible prompts ask for a contradiction, which cannot be scored, so the task here stays solvable and the pressure is in the wrapper. Neutral has an empty prefix. The task text is the whole control, and it is a suffix of every other prompt.

`EMOTION_QUICK=1` keeps the first two items of each kind. That order is the file order, not a random draw.


In [ ]:
tasks = load_tasks()
framings = load_framings()
if cfg.quick:
    from llm_emotions.phase03 import select_tasks

    tasks = select_tasks(tasks, quick=True, per_kind=cfg.quick_items_per_kind)
prompts = build_phase03_prompts(tasks, framings)
print(f"items = {len(tasks)}")
print(f"framings = {[row.id for row in framings]}")
print(f"prompts = {len(prompts)}")
print("\nneutral and insult for the first item:")
print(prompts[0].prompt)
print("---")
print(prompts[1].prompt)


## 2. Load the model

Residuals are read at the phase-02 layer. Index 0 is the embedding stream. The reply is the greedy continuation, stopped at eos or at `max_new_tokens`, whichever comes first.


In [ ]:
loaded = load_model_and_tokenizer(cfg.model_id, trust_remote_code=cfg.trust_remote_code)
model, tokenizer = loaded.model, loaded.tokenizer
print(f"device = {loaded.device}, dtype = {loaded.dtype}")
print(f"layers = {loaded.n_layers}, hidden_size = {loaded.hidden_size}")


## 3. Score the replies and relate them to the anger projection

The runner writes `results/phase03_performance/<model>/`.

For the default 1.5B model it loads the committed phase-02 directions when `results/phase02_scenarios/<model>/directions.npz` is present, and otherwise the packaged layer-12 slice. A different model id, or a hidden size that does not match, refits the cleaned directions with the phase-02 rule before any reply is generated.

Each reply records correctness, word count, the four marker flags, and the projection of the prompt and of the reply onto every cleaned direction. `prompt_anger_minus_neutral` is the anger projection minus the neutral wording of the same item. The paired accuracy gap resamples items. McNemar's exact test is the discordant pairs. The partial association called `framing` removes the framing mean. `framing_and_item` also removes the item mean. Both are correlational. The bootstrap is not corrected for the grid.


In [ ]:
result = run_phase03(cfg, model=model, tokenizer=tokenizer)
print(format_phase03_summary(result.metrics))


In [ ]:
from IPython.display import display

for name, fig in result.figures.items():
    print(name)
    display(fig)


## 4. How to read the files

`replies.csv` is one row per item and framing. `framing_means.csv` is the mean of each outcome inside a framing, with a 95% bootstrap interval across items. `framing_effects.csv` is that outcome minus the neutral wording of the same item. `associations.csv` is the anger projection against correctness, length, and the marker flags. `examples.json` keeps the first item of each kind under every framing. The three figures are the accuracy bars, the marker rates, and the residual scatter.

A quick run is labelled `"quick": true` in `metrics.json`. Do not quote it as the study. A full run is still one wording per framing, one model, and an uncorrected set of intervals. Phase 04 would have to steer the direction before a change in accuracy could be attributed to it.


## 5. Download the results

Colab cannot download a folder. This cell zips `results/` (this run, and a phase-01 or phase-02 run too if those notebooks were used in the same session) and starts a file download.


In [ ]:
import shutil

archive_path = shutil.make_archive("llm-emotions-results", "zip", root_dir=".", base_dir="results")
print(archive_path)
try:
    from google.colab import files

    files.download(archive_path)
except ImportError:
    print("Not running in Colab; the zip is in the working directory.")
